# <font color="#418FDE" size="6.5" uppercase>**Images Custom**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Represent images as arrays and extract patch-based features. 
- Create graph structures for image neighborhoods and regular grids. 
- Implement simple custom transformers compatible with scikit-learn pipelines. 


## **1. Image Patch Arrays**

### **1.1. Image Array Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_01.jpg?v=1783955237" width="250">



>* Images are numerical pixel arrays
>* Color images stack separate channel grids

>* Nearby pixels reveal local image patterns
>* Patches turn neighborhoods into measurable features

>* Check size, channels, values, and data types
>* Arrays turn images into measurable features



In [ ]:
#@title Python Code - Image Array Basics

# Images are arrays of numbers.
# Small patches reveal local structure.
# Shapes describe height, width, channels.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny grayscale image array.
image = np.array([
    [0, 20, 40, 60, 80, 100],
    [10, 30, 50, 70, 90, 110],

    [20, 40, 200, 220, 120, 130],
    [30, 50, 210, 230, 130, 140],
    [40, 60, 80, 100, 150, 160],
    [50, 70, 90, 110, 170, 180],
], dtype=np.uint8)

# Validate the expected two dimensional shape.
assert image.ndim == 2 and image.shape == (6, 6)

# Select one local rectangular patch.
patch = image[2:4, 2:4]
assert patch.shape == (2, 2)

# Summarize the image and selected patch.
print(f"Image shape: {image.shape}")
print(f"Image dtype: {image.dtype}")
print(f"Pixel range: {image.min()} to {image.max()}")
print(f"Patch shape: {patch.shape}")
print(f"Patch mean intensity: {patch.mean():.1f}")

# Display the image and highlight the patch.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image, cmap="gray", vmin=0, vmax=255)

# Draw a red box around the patch.
box = plt.Rectangle((1.5, 1.5), 2, 2, fill=False, color="red", linewidth=3)
ax.add_patch(box)

# Label axes as array coordinates.
ax.set_title("Grayscale image array with one patch")
ax.set_xlabel("Column index")
ax.set_ylabel("Row index")

# Show pixel grid positions clearly.
ax.set_xticks(range(image.shape[1]))
ax.set_yticks(range(image.shape[0]))
ax.grid(color="white", linewidth=0.5)

plt.show()



### **1.2. Patch Extraction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_02.jpg?v=1783955240" width="250">



>* Split images into smaller local patches
>* Use patches for structured visual features

>* Choose overlapping or separate patch positions
>* Balance patch size for detail and context

>* Flatten patches into machine learning features
>* Connect pixels to higher-level visual patterns



In [ ]:
#@title Python Code - Patch Extraction

# Patch extraction turns images into local arrays.
# Small patches reveal edges and textures.
# Flattened patches become simple feature vectors.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny grayscale image array.
image = np.arange(64, dtype=float).reshape(8, 8)
image[2:6, 3:5] += 40

# Choose patch size and stride.
patch_height, patch_width = 3, 3
stride_rows, stride_cols = 2, 2

# Validate patch settings before extraction.
assert image.ndim == 2, "Use one grayscale image."
assert patch_height <= image.shape[0], "Patch height is too large."
assert patch_width <= image.shape[1], "Patch width is too large."

# Extract overlapping patches from the image.
patches = []
positions = []
for row in range(0, image.shape[0] - patch_height + 1, stride_rows):
    for col in range(0, image.shape[1] - patch_width + 1, stride_cols):
        patch = image[row:row + patch_height, col:col + patch_width]
        patches.append(patch)
        positions.append((row, col))

# Stack patches into one array.
patch_array = np.stack(patches, axis=0)
flat_patches = patch_array.reshape(patch_array.shape[0], -1)

# Print compact shape information only.
print("Image shape:", image.shape)
print("Patch array shape:", patch_array.shape)
print("Flattened feature shape:", flat_patches.shape)
print("First patch position:", positions[0])
print("First patch mean:", round(float(patch_array[0].mean()), 2))

# Show the image and extracted patches.
fig, axes = plt.subplots(1, 5, figsize=(10, 2.4))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Image")
axes[0].axis("off")

# Display four example patches.
for axis, patch, position in zip(axes[1:], patch_array[:4], positions[:4]):
    axis.imshow(patch, cmap="gray")
    axis.set_title(f"{position}")
    axis.axis("off")

# Keep the single plot readable.
plt.tight_layout()
plt.show()



### **1.3. Patch Reconstruction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_03.jpg?v=1783955242" width="250">



>* Reassemble patches into the full image
>* Preserve locations after patch-based processing

>* Overlapping patches preserve boundary details
>* Averaging overlaps reduces seams and blockiness

>* Rebuild patches into meaningful full images
>* Map patch scores for visual interpretation



In [ ]:
#@title Python Code - Patch Reconstruction

# Rebuild images from small patches.
# Overlapping patches can be averaged.
# This example uses tiny arrays.

import numpy as np
import matplotlib.pyplot as plt

# Create a small image-like array.
image = np.arange(64, dtype=float).reshape(8, 8)
patch_size = 4
step_size = 2

# Validate patch settings before extraction.
height, width = image.shape
assert patch_size <= height and patch_size <= width
assert step_size > 0 and patch_size > 0

# Store patches and their positions.
patches = []
positions = []
for row in range(0, height - patch_size + 1, step_size):
    for col in range(0, width - patch_size + 1, step_size):
        patches.append(image[row:row + patch_size, col:col + patch_size])
        positions.append((row, col))

# Reconstruct by summing overlapping patch values.
reconstructed = np.zeros_like(image)
counts = np.zeros_like(image)
for patch, (row, col) in zip(patches, positions):
    reconstructed[row:row + patch_size, col:col + patch_size] += patch
    counts[row:row + patch_size, col:col + patch_size] += 1

# Average pixels that received multiple patches.
safe_counts = np.maximum(counts, 1)
reconstructed = reconstructed / safe_counts
max_error = np.abs(image - reconstructed).max()

# Print a compact summary for learners.
print(f"Original image shape: {image.shape}")
print(f"Extracted patches: {len(patches)}")
print(f"Each patch shape: {patches[0].shape}")
print(f"Maximum reconstruction error: {max_error:.1f}")

# Show original, overlap counts, and reconstruction.
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image, cmap="viridis")
axes[0].set_title("Original")
axes[1].imshow(counts, cmap="magma")
axes[1].set_title("Overlap counts")
axes[2].imshow(reconstructed, cmap="viridis")
axes[2].set_title("Reconstructed")

# Hide axes for a cleaner teaching plot.
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()



## **2. Image Graphs**

### **2.1. Pixel Neighborhood Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_01.jpg?v=1783955223" width="250">



>* Pixels become nodes connected to nearby pixels
>* Local relationships reveal visual patterns and features

>* Neighborhood choice shapes captured pixel interactions
>* Wider neighborhoods reveal textures and spatial patterns

>* Preserve image structure through pixel relationships
>* Use edge weights to reveal visual features



In [ ]:
#@title Python Code - Pixel Neighborhood Graphs

# Build a tiny image graph.
# Compare four and eight neighbors.
# Visualize pixels as connected nodes.

import numpy as np
import matplotlib.pyplot as plt

# Create a small grayscale image.
image = np.array([
    [0.1, 0.2, 0.8, 0.9],
    [0.1, 0.3, 0.7, 0.9],
    [0.2, 0.4, 0.6, 0.8],
    [0.3, 0.4, 0.5, 0.7],
])

# Validate the image shape.
height, width = image.shape
assert height == 4 and width == 4

# Convert row and column into node id.
def node_id(row, col, width):
    return row * width + col

# Build edges for chosen neighborhood.
def build_edges(height, width, diagonal=False):
    offsets = [(0, 1), (1, 0)]
    if diagonal:
        offsets += [(1, 1), (1, -1)]
    edges = []
    for row in range(height):
        for col in range(width):
            for dr, dc in offsets:
                nr, nc = row + dr, col + dc
                if 0 <= nr < height and 0 <= nc < width:
                    a = node_id(row, col, width)
                    b = node_id(nr, nc, width)
                    edges.append((a, b))
    return edges

# Create two neighborhood graphs.
four_edges = build_edges(height, width, diagonal=False)
eight_edges = build_edges(height, width, diagonal=True)

# Compute simple contrast weights.
def edge_weight(edge, image, width):
    a, b = edge
    ar, ac = divmod(a, width)
    br, bc = divmod(b, width)
    return abs(float(image[ar, ac] - image[br, bc]))

# Summarize graph sizes and contrasts.
four_weights = [edge_weight(e, image, width) for e in four_edges]
eight_weights = [edge_weight(e, image, width) for e in eight_edges]

print(f"Image shape: {height} by {width} pixels")
print(f"Nodes: {height * width}")
print(f"Four-neighbor edges: {len(four_edges)}")
print(f"Eight-neighbor edges: {len(eight_edges)}")
print(f"Mean four-neighbor contrast: {np.mean(four_weights):.3f}")
print(f"Mean eight-neighbor contrast: {np.mean(eight_weights):.3f}")

# Prepare node positions for plotting.
positions = {}
for row in range(height):
    for col in range(width):
        positions[node_id(row, col, width)] = (col, height - 1 - row)

# Draw the image graph.
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image, cmap="gray", origin="upper")

# Draw eight-neighbor edges lightly.
for a, b in eight_edges:
    x1, y1 = positions[a]
    x2, y2 = positions[b]
    ax.plot([x1, x2], [y1, y2], color="cyan", alpha=0.45)

# Draw pixel nodes clearly.
for node, (x, y) in positions.items():
    ax.scatter(x, y, s=120, color="orange", edgecolor="black")
    ax.text(x, y, str(node), ha="center", va="center")

# Format the single plot.
ax.set_title("Eight-neighbor pixel graph on a 4x4 image")
ax.set_xticks(range(width))
ax.set_yticks(range(height))
ax.set_xlim(-0.5, width - 0.5)
ax.set_ylim(-0.5, height - 0.5)

# Show the graph visualization.
plt.show()



### **2.2. Regular Grid Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_02.jpg?v=1783955220" width="250">



>* Pixels become nodes linked to neighbors
>* Grid structure captures spatial image relationships

>* Grid graphs preserve image neighborhood structure
>* Connectivity and weights guide boundary-aware relationships

>* Connect image arrays with graph-based analysis
>* Capture spatial context for visual tasks



In [ ]:
#@title Python Code - Regular Grid Graphs

# Build a tiny image grid.
# Connect pixels to nearby neighbors.
# Visualize regular grid graph edges.

import numpy as np
import matplotlib.pyplot as plt

# Create a small grayscale image.
height, width = 5, 6
image = np.arange(height * width).reshape(height, width)

# Validate the image shape.
assert image.ndim == 2
assert height * width == image.size

# Convert row and column into node id.
def node_id(row, col):
    return row * width + col

# Build four-neighbor grid edges.
edges_four = []
for row in range(height):
    for col in range(width):
        if col + 1 < width:
            edges_four.append((node_id(row, col), node_id(row, col + 1)))
        if row + 1 < height:
            edges_four.append((node_id(row, col), node_id(row + 1, col)))

# Build eight-neighbor diagonal edges.
diagonal_edges = []
for row in range(height - 1):
    for col in range(width - 1):
        diagonal_edges.append((node_id(row, col), node_id(row + 1, col + 1)))
        diagonal_edges.append((node_id(row + 1, col), node_id(row, col + 1)))

# Combine four and diagonal neighborhoods.
edges_eight = edges_four + diagonal_edges
center_node = node_id(2, 3)

# Find neighbors touching the center node.
center_neighbors = sorted({b if a == center_node else a for a, b in edges_eight if center_node in (a, b)})

# Print a compact graph summary.
print(f"Image shape: {height} rows by {width} columns")
print(f"Nodes: {image.size}")
print(f"Four-neighbor edges: {len(edges_four)}")
print(f"Eight-neighbor edges: {len(edges_eight)}")
print(f"Center node {center_node} neighbors: {center_neighbors}")

# Prepare node coordinates for plotting.
coords = {node_id(r, c): (c, -r) for r in range(height) for c in range(width)}
fig, ax = plt.subplots(figsize=(6, 4))

# Draw four-neighbor edges first.
for a, b in edges_four:
    x1, y1 = coords[a]
    x2, y2 = coords[b]
    ax.plot([x1, x2], [y1, y2], color="steelblue", linewidth=1.5)

# Draw diagonal edges lightly.
for a, b in diagonal_edges:
    x1, y1 = coords[a]
    x2, y2 = coords[b]
    ax.plot([x1, x2], [y1, y2], color="orange", linewidth=0.8, alpha=0.6)

# Draw nodes over the edges.
xs = [coords[n][0] for n in coords]
ys = [coords[n][1] for n in coords]
ax.scatter(xs, ys, s=120, color="white", edgecolor="black", zorder=3)

# Label each node with its id.
for n, (x, y) in coords.items():
    ax.text(x, y, str(n), ha="center", va="center", fontsize=8)

# Format the graph display.
ax.set_title("Regular Grid Graph: 4-neighbor plus diagonals")
ax.set_aspect("equal")
ax.axis("off")
plt.show()



### **2.3. Graph Applications**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_03.jpg?v=1783955225" width="250">



>* Model image parts and their connections
>* Use neighborhoods to segment meaningful regions

>* Smooth noisy images while preserving edges
>* Spread labels using neighborhood relationships

>* Graph features support machine learning pipelines
>* Spatial relationships improve visual understanding



## **3. Custom Transformers**

### **3.1. Function Based Transforms**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_01.jpg?v=1783955232" width="250">



>* Wrap image preprocessing functions as pipeline steps
>* Apply transformations consistently across model stages

>* Use fixed transforms for predictable image conversions
>* Test domain features inside pipelines quickly

>* Design transforms for consistent image shapes
>* Use fitted transformers when learning parameters



In [ ]:
#@title Python Code - Function Based Transforms

# Function transforms wrap simple image feature ideas.
# Tiny arrays keep this example beginner friendly.
# Pipeline style steps can stay reproducible.

import numpy as np

# Create two tiny grayscale images.
images = np.array([
    [[0, 40, 80, 120], [20, 60, 100, 140], [40, 80, 120, 160], [60, 100, 140, 180]],
    [[180, 140, 100, 60], [160, 120, 80, 40], [140, 100, 60, 20], [120, 80, 40, 0]]
], dtype=float)

# Validate the expected image batch shape.
if images.ndim != 3:
    raise ValueError("Expected batch, height, width array")

# Normalize pixel values into zero one range.
def normalize_pixels(batch):
    batch = np.asarray(batch, dtype=float)
    return batch / 255.0

# Extract simple patch means from each image.
def patch_mean_features(batch, patch_size=2):
    batch = np.asarray(batch, dtype=float)
    n_images, height, width = batch.shape

    # Ensure patches divide the image exactly.
    if height % patch_size or width % patch_size:
        raise ValueError("Patch size must divide image dimensions")

    # Reshape images into nonoverlapping square patches.
    patches = batch.reshape(
        n_images, height // patch_size, patch_size,
        width // patch_size, patch_size
    )

    # Average each patch into one feature.
    means = patches.mean(axis=(2, 4))
    return means.reshape(n_images, -1)

# Apply functions like pipeline transform steps.
normalized_images = normalize_pixels(images)
feature_matrix = patch_mean_features(normalized_images)

# Show compact results without printing large arrays.
print("Input image batch shape:", images.shape)
print("Feature matrix shape:", feature_matrix.shape)
print("First image patch features:", np.round(feature_matrix[0], 3))
print("Second image patch features:", np.round(feature_matrix[1], 3))



### **3.2. TransformerMixin Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_02.jpg?v=1783955227" width="250">



>* Custom transformers learn from training data
>* Fit then transform within scikit-learn pipelines

>* Separate user settings from learned attributes
>* Keep transformers stable inside pipelines

>* Custom transformers fit smoothly into pipelines
>* Consistent steps reduce leakage and maintenance



### **3.3. Feature Name Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_03.jpg?v=1783955235" width="250">



>* Preserve meaning through image feature transformations
>* Name columns for debugging and explanation

>* Keep feature names consistent across pipeline stages
>* Link model outputs to real image properties

>* Plan input and output feature names
>* Clear names make pipelines easier to audit



In [ ]:
#@title Python Code - Feature Name Handling

# Custom transformers should preserve feature meaning.
# This example uses tiny image patches.
# Feature names travel with transformed columns.

import numpy as np
import pandas as pd

# Define a small scikit-learn style transformer.
class PatchMeanTransformer:
    def __init__(self, patch_size=(2, 2)):
        self.patch_size = patch_size

    def fit(self, images, y=None):
        images = np.asarray(images)
        if images.ndim != 3:
            raise ValueError("Expected shape: samples, height, width")

        self.image_shape_ = images.shape[1:]
        self.feature_names_ = self._make_names()
        return self

    def _make_names(self):
        height, width = self.image_shape_
        patch_h, patch_w = self.patch_size

        names = []
        for row in range(0, height, patch_h):
            for col in range(0, width, patch_w):
                names.append(f"mean_r{row}_c{col}")

        return np.array(names, dtype=object)

    def transform(self, images):
        images = np.asarray(images)
        if images.shape[1:] != self.image_shape_:
            raise ValueError("New images must match fitted shape")

        rows = []
        patch_h, patch_w = self.patch_size
        for image in images:
            values = []
            for row in range(0, image.shape[0], patch_h):
                for col in range(0, image.shape[1], patch_w):
                    patch = image[row:row+patch_h, col:col+patch_w]
                    values.append(float(patch.mean()))

            rows.append(values)

        return np.array(rows)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_.copy()

# Create two tiny grayscale images.
images = np.array([
    [[0, 0, 8, 8], [0, 0, 8, 8], [4, 4, 9, 9], [4, 4, 9, 9]],
    [[1, 1, 7, 7], [1, 1, 7, 7], [5, 5, 6, 6], [5, 5, 6, 6]],
])

# Fit, transform, and attach readable names.
transformer = PatchMeanTransformer(patch_size=(2, 2))
features = transformer.fit(images).transform(images)
names = transformer.get_feature_names_out()

# Build a compact table for inspection.
feature_table = pd.DataFrame(features, columns=names)
print("Feature names:", list(names))
print("Feature matrix shape:", feature_table.shape)
print(feature_table.round(2).to_string(index=False))



# <font color="#418FDE" size="6.5" uppercase>**Images Custom**</font>


In this lecture, you learned to:
- Represent images as arrays and extract patch-based features. 
- Create graph structures for image neighborhoods and regular grids. 
- Implement simple custom transformers compatible with scikit-learn pipelines. 

In the next Module (Module 8), we will go over 'Kernels Approximation'